# North San Andreas fault M7.6 Scenario

## 1. Setup

In [ ]:
import os, json, shutil
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import base64
from IPython.display import HTML

EVENT = 'bssc2014nsanandreassapsassha_m7p6_se'
os.environ['EVENT'] = EVENT

EVENT_DIR = Path.home() / 'shakemap_profiles/default/data' / EVENT / 'current'
PRODUCTS  = EVENT_DIR / 'products'
GF_OUT    = Path.home() / 'gf_output' / EVENT
CA_DATA   = Path('/workspaces/shakemap-codespaces/data/california_inputs')
PAGER_PNG = Path('/workspaces/shakemap-codespaces/data/pager/onepager.png')

## 2. Run ShakeMap

ShakeMap combines three types of input to estimate ground shaking:
- **Earthquake origin** — location, magnitude, depth, and fault geometry (`event.xml` + `rupture.json`)
- **Ground motion models (GMMs)** — equations that predict shaking from earthquakes
- **Site conditions (Vs30)** — local geology that amplifies or de-amplifies shaking (soft soils shake more than rock)

Four modules run in sequence:
- **`assemble`** — collect and validate all inputs
- **`model`** — compute ground motion at every grid point (computationally intense; about 1 minute to finish)
- **`contour`** — generate MMI shaking contour lines
- **`mapping`** — render map images (can take 30-45s)
- **`gridxml`** — export the grid file needed for ground failure models

In [ ]:
%%bash
shake $EVENT assemble -c 'demo' model contour mapping gridxml

## 3. ShakeMap maps

We will plot the **Modified Mercalli Intensity (MMI)** map — which
describes felt shaking in human terms — and the **Peak Ground Acceleration (PGA)**
map, which is used in engineering design.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, name, title in [
    (axes[0], 'intensity.jpg', 'MMI Intensity'),
    (axes[1], 'pga.jpg',       'PGA (%g)'),
]:
    img_path = PRODUCTS / name
    if img_path.exists():
        ax.imshow(mpimg.imread(img_path))
    ax.set_title(title, fontsize=13); ax.axis('off')
plt.suptitle('M7.6 North San Andreas', fontsize=14)
plt.tight_layout(); plt.show()

shutil.copy(PRODUCTS / 'intensity.jpg',
            Path.home() / 'san_andreas_ff_intensity.jpg')

## 4. PAGER — Loss Estimates

PAGER estimates fatalities and economic losses using ShakeMap grids combined with
with population data (LandScan) and country-level building vulnerability models.

Note the separate alerts: California has earthquake-resistant construction
(yellow fatalities) but extremely high property values (red economic losses).

In [ ]:
img = mpimg.imread(str(PAGER_PNG))
h, w = img.shape[:2]
fig, ax = plt.subplots(figsize=(w/150, h/150))
ax.imshow(img, interpolation='lanczos')
ax.axis('off')
plt.tight_layout(pad=0)
plt.show()

## 5. Ground failure

Ground failure models estimate secondary hazards triggered by the shaking:
- **Landslide probability** — Nowicki Jessee et al. (2018): uses slope, PGV, and susceptibility
- **Liquefaction probability** — Zhu et al. (2017): uses Vs30, distance to water, and precipitation

The Santa Cruz Mountains produce high landslide hazard; Bay mud deposits
along the bay margins produce high liquefaction hazard.

Note These models run in the `gf` conda environment — we call them by their full
path so we don't need to switch environments mid-notebook.

In [ ]:
%%bash
/opt/conda/envs/gf/bin/gfailbin \
  ~/groundfailure/defaultconfigfiles/models/jessee_2018_slim.ini \
  ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --gis -d /workspaces/shakemap-codespaces/data/california_inputs

In [ ]:
%%bash
/opt/conda/envs/gf/bin/gfailbin \
  ~/groundfailure/defaultconfigfiles/models/zhu_2017_general_slim.ini \
  ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --gis -d /workspaces/shakemap-codespaces/data/california_inputs \

## 6. Interactive ground failure map

The map below shows landslide (left) and liquefaction (right) probability
side by side, with shaking contours overlaid.

In [ ]:
%%bash
/opt/conda/envs/gf/bin/python ~/plot_gf_interactive.py \
  --ls-model "Jessee 2018:$HOME/gf_output/$EVENT/${EVENT}_jessee_2018_slim_model.tif:$HOME/groundfailure/defaultconfigfiles/models/jessee_2018_slim.ini" \
  --lq-model "Zhu 2017:$HOME/gf_output/$EVENT/${EVENT}_zhu_2017_general_slim_model.tif:$HOME/groundfailure/defaultconfigfiles/models/zhu_2017_general_slim.ini" \
  --shakefile ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --contours  ~/shakemap_profiles/default/data/$EVENT/current/products/cont_mmi.json \
  --outfile ~/san_andreas_gf.html 2>&1 | tail -3

In [ ]:
html_path = Path.home() / 'san_andreas_gf.html'
b64 = base64.b64encode(html_path.read_bytes()).decode()
HTML(f'<iframe src="data:text/html;base64,{b64}" width="100%" height="600px"></iframe>')

---